In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Environment Ready")

Environment Ready


In [2]:
import pandas as pd

df = pd.read_csv(r"C:\Users\Reena\OneDrive\Documents\known_exploited_vulnerabilities.csv")

print(df.head())
print(df.shape)
print(df.columns)

            cveID        vendorProject  \
0  CVE-2026-22719             Broadcom   
1  CVE-2026-21385             Qualcomm   
2  CVE-2022-20775                Cisco   
3  CVE-2026-20127                Cisco   
4  CVE-2026-25108  Soliton Systems K.K   

                                  product  \
0                  VMware Aria Operations   
1                       Multiple Chipsets   
2                                  SD-WAN   
3  Catalyst SD-WAN Controller and Manager   
4                                 FileZen   

                                   vulnerabilityName   dateAdded  \
0  Broadcom VMware Aria Operations Command Inject...  03-03-2026   
1  Qualcomm Multiple Chipsets Memory Corruption V...  03-03-2026   
2          Cisco SD-WAN Path Traversal Vulnerability  25-02-2026   
3  Cisco Catalyst SD-WAN Controller and Manager A...  25-02-2026   
4  Soliton Systems K.K FileZen OS Command Injecti...  24-02-2026   

                                    shortDescription  \
0  Broadcom

In [3]:
print(df.shape)
print(df.columns)
df.head()

(1531, 11)
Index(['cveID', 'vendorProject', 'product', 'vulnerabilityName', 'dateAdded',
       'shortDescription', 'requiredAction', 'dueDate',
       'knownRansomwareCampaignUse', 'notes', 'cwes'],
      dtype='object')


,cveID,vendorProject,product,vulnerabilityName,dateAdded,shortDescription,requiredAction,dueDate,knownRansomwareCampaignUse,notes,cwes
0,CVE-2026-22719,Broadcom,VMware Aria Operations,Broadcom VMware Aria Operations Command Inject...,03-03-2026,Broadcom VMware Aria Operations formerly known...,"Apply mitigations per vendor instructions, fol...",24-03-2026,Unknown,https://support.broadcom.com/web/ecx/support-c...,CWE-77
1,CVE-2026-21385,Qualcomm,Multiple Chipsets,Qualcomm Multiple Chipsets Memory Corruption V...,03-03-2026,Multiple Qualcomm chipsets contain a memory co...,"Apply mitigations per vendor instructions, fol...",24-03-2026,Unknown,https://source.android.com/docs/security/bulle...,CWE-190
2,CVE-2022-20775,Cisco,SD-WAN,Cisco SD-WAN Path Traversal Vulnerability,25-02-2026,Cisco SD-WAN CLI contains a path traversal vul...,Please adhere to CISA’s guidelines to assess e...,27-02-2026,Unknown,CISA Mitigation Instructions: https://www.cisa...,"CWE-25, CWE-282"
3,CVE-2026-20127,Cisco,Catalyst SD-WAN Controller and Manager,Cisco Catalyst SD-WAN Controller and Manager A...,25-02-2026,"Cisco Catalyst SD-WAN Controller, formerly SD-...",Please adhere to CISA’s guidelines to assess e...,27-02-2026,Unknown,CISA Mitigation Instructions: https://www.cisa...,CWE-287
4,CVE-2026-25108,Soliton Systems K.K,FileZen,Soliton Systems K.K FileZen OS Command Injecti...,24-02-2026,Soliton Systems K.K FileZen contains an OS com...,"Apply mitigations per vendor instructions, fol...",17-03-2026,Unknown,https://jvn.jp/en/jp/JVN84622767/ ; https://nv...,CWE-78


In [4]:
df['dateAdded'] = pd.to_datetime(df['dateAdded'], dayfirst=True)
df['dueDate'] = pd.to_datetime(df['dueDate'], dayfirst=True)

In [5]:
df[['dateAdded','dueDate']].head()

,dateAdded,dueDate
0,2026-03-03,2026-03-24
1,2026-03-03,2026-03-24
2,2026-02-25,2026-02-27
3,2026-02-25,2026-02-27
4,2026-02-24,2026-03-17


In [6]:
df['year'] = df['dateAdded'].dt.year
df['month'] = df['dateAdded'].dt.month

In [7]:
df['patch_delay_days'] = (df['dueDate'] - df['dateAdded']).dt.days

In [8]:
df['patch_delay_days'].describe()

count    1531.000000
mean       46.517309
std        60.589578
min         1.000000
25%        21.000000
50%        21.000000
75%        21.000000
max       184.000000
Name: patch_delay_days, dtype: float64

In [9]:
def risk_level(x):
    if x <= 7:
        return "Low"    
    
    elif x <= 30:
        return "Medium"
    else:
        return "High"

In [10]:
df['risk_level'] = df['patch_delay_days'].apply(risk_level)

In [11]:
df.to_csv("cyber_threat_intelligence_dataset.csv", index=False)

In [12]:
# =====================================================
# PROFESSIONAL ML PIPELINE - NO DATA LEAKAGE
# Target: Predict if a vulnerability is linked to Known Ransomware Campaign
# This is REAL cyber threat intelligence
# =====================================================

# 1. Create REAL target (what analysts actually care about)
df['ransomware_risk'] = df['knownRansomwareCampaignUse'].apply(lambda x: 1 if str(x).strip().lower() == 'known' else 0)

# 2. Features that are available BEFORE the attack (NO patch_delay_days!)
features = df[['vendorProject', 'product', 'year', 'month', 'cwes']].copy()
target = df['ransomware_risk']

# 3. Encode categorical columns
from sklearn.preprocessing import LabelEncoder

le_vendor = LabelEncoder()
le_product = LabelEncoder()
le_cwes = LabelEncoder()

features['vendorProject'] = le_vendor.fit_transform(features['vendorProject'].astype(str))
features['product'] = le_product.fit_transform(features['product'].astype(str))
features['cwes'] = le_cwes.fit_transform(features['cwes'].astype(str))

# 4. Train-Test Split (stratified because ransomware cases are rare)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.2, random_state=42, stratify=target
)

# 5. Train Random Forest with balanced classes
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight='balanced',
    max_depth=10,
    min_samples_leaf=2
)
model.fit(X_train, y_train)

# 6. REAL Evaluation (this will NOT be 1.0 anymore)
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

predictions = model.predict(X_test)
probabilities = model.predict_proba(X_test)[:, 1]

print("=== MODEL PERFORMANCE ===")
print("Accuracy          :", round(accuracy_score(y_test, predictions), 4))
print("ROC-AUC Score     :", round(roc_auc_score(y_test, probabilities), 4))
print("\nClassification Report:\n", classification_report(y_test, predictions))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, predictions))

# 7. Feature Importance (Very Important for Dashboard)
import pandas as pd
importances = pd.DataFrame({
    'Feature': ['vendorProject', 'product', 'year', 'month', 'cwes'],
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\n=== TOP FEATURES ===")
print(importances)

# 8. Add predictions back to original dataframe for Power BI
df['predicted_ransomware_risk'] = model.predict(features)
df['ransomware_probability'] = model.predict_proba(features)[:, 1]

# 9. Export clean file for Power BI
df.to_csv("cyber_threat_ml_output.csv", index=False)
print("\n✅ SUCCESS: cyber_threat_ml_output.csv created with ML predictions!")
print("Columns added: predicted_ransomware_risk + ransomware_probability")

=== MODEL PERFORMANCE ===
Accuracy          : 0.7557
ROC-AUC Score     : 0.7519

Classification Report:
               precision    recall  f1-score   support

           0       0.85      0.84      0.85       244
           1       0.41      0.43      0.42        63

    accuracy                           0.76       307
   macro avg       0.63      0.63      0.63       307
weighted avg       0.76      0.76      0.76       307


Confusion Matrix:
 [[205  39]
 [ 36  27]]

=== TOP FEATURES ===
         Feature  Importance
1        product    0.285689
0  vendorProject    0.261440
4           cwes    0.201855
3          month    0.128284
2           year    0.122732

✅ SUCCESS: cyber_threat_ml_output.csv created with ML predictions!
Columns added: predicted_ransomware_risk + ransomware_probability


In [13]:
# === CREATE PROPER EXCEL FILES FOR POWER BI ===

import pandas as pd

# Re-create the three tables again (clean version)
powerbi_main = df[[
    'cveID', 'vendorProject', 'product', 'vulnerabilityName',
    'dateAdded', 'dueDate', 'shortDescription', 'requiredAction',
    'knownRansomwareCampaignUse', 'cwes', 'patch_delay_days', 
    'risk_level', 'predicted_ransomware_risk', 'ransomware_probability', 'year', 'month'
]].copy()

vendor_summary = powerbi_main.groupby('vendorProject').agg(
    total_vulnerabilities=('cveID', 'count'),
    ransomware_linked=('predicted_ransomware_risk', 'sum'),
    avg_risk_probability=('ransomware_probability', 'mean'),
    high_risk_count=('predicted_ransomware_risk', lambda x: (x == 1).sum()),
    avg_patch_delay=('patch_delay_days', 'mean')
).reset_index()

vendor_summary['ransomware_percentage'] = (vendor_summary['ransomware_linked'] / vendor_summary['total_vulnerabilities'] * 100).round(2)

monthly_trend = powerbi_main.groupby(['year', 'month']).agg(
    new_vulnerabilities=('cveID', 'count'),
    ransomware_cases=('predicted_ransomware_risk', 'sum')
).reset_index()

monthly_trend['date'] = pd.to_datetime(monthly_trend['year'].astype(str) + '-' + monthly_trend['month'].astype(str).str.zfill(2) + '-01')

# === SAVE AS PROPER EXCEL FILES ===
powerbi_main.to_excel("powerbi_main.xlsx", index=False)
vendor_summary.to_excel("vendor_risk_summary.xlsx", index=False)
monthly_trend.to_excel("monthly_trend.xlsx", index=False)

print("✅ THREE CLEAN EXCEL FILES CREATED!")
print("Files saved:")
print("1. powerbi_main.xlsx")
print("2. vendor_risk_summary.xlsx")
print("3. monthly_trend.xlsx")
print("\nNow these are proper Excel files. You can open them in Excel and everything will be in separate columns.")

✅ THREE CLEAN EXCEL FILES CREATED!
Files saved:
1. powerbi_main.xlsx
2. vendor_risk_summary.xlsx
3. monthly_trend.xlsx

Now these are proper Excel files. You can open them in Excel and everything will be in separate columns.
